# Validação cruzada: a régua que decide se o número vale

Todos os notebooks anteriores mediram com validação cruzada e nenhum explicou por quê. Este explica, e
faz isso medindo: cada decisão de validação errada tem um preço, e o preço aparece no número.

São cinco decisões, e cada uma tem uma situação em que ela decide o resultado:

1. **uma divisão só** ou várias;
2. **embaralhar** ou não, e **estratificar** ou não;
3. as linhas são **independentes** ou existem grupos dentro delas;
4. existe **ordem** no tempo;
5. o número que você escolheu o modelo é o **mesmo** que você reporta.


In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import (GridSearchCV, GroupKFold, KFold, StratifiedKFold,
                                     TimeSeriesSplit, cross_val_score, cross_validate,
                                     train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind",
              rc={"figure.figsize": (9, 4)})

RANDOM_STATE = 42

dados = load_breast_cancer()
X = pd.DataFrame(dados.data, columns=dados.feature_names)
y = pd.Series(dados.target, name="maligno")
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

print(f"tabela: {X.shape[0]} linhas x {X.shape[1]} colunas")
print(f"alvo: {y.mean():.1%} de classe positiva")
print("as 20 primeiras linhas do alvo, para ver a ordem em que ele vem:")
print(f"  {y.to_numpy()[:20]}")
print("as 5 ultimas:")
print(f"  {y.to_numpy()[-5:]}")


Repare na ordem do alvo: as classes estão agrupadas, não intercaladas. Isso vai importar no segundo
experimento.

## Uma divisão só mente

A forma mais comum de medir é dividir uma vez, treinar, olhar o teste e anotar o número. O problema é que
esse número depende do sorteio da divisão, e ninguém reporta o sorteio. Rodando a mesma medição com vinte
sementes diferentes, o quanto o número se mexe?


In [ ]:
notas_de_uma_divisao = []
for semente in range(20):
    X_treino, X_teste, y_treino, y_teste = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=semente)
    modelo.fit(X_treino, y_treino)
    notas_de_uma_divisao.append(roc_auc_score(y_teste, modelo.predict_proba(X_teste)[:, 1]))

notas_de_uma_divisao = pd.Series(notas_de_uma_divisao, name="roc_auc")
print(f"menor: {notas_de_uma_divisao.min():.4f} | mediana: {notas_de_uma_divisao.median():.4f} "
      f"| maior: {notas_de_uma_divisao.max():.4f}")
print(f"amplitude entre a pior e a melhor semente: {notas_de_uma_divisao.max() - notas_de_uma_divisao.min():.4f}")

sns.boxplot(notas_de_uma_divisao, color="lightsteelblue")
sns.stripplot(notas_de_uma_divisao, color="black", size=5)
plt.title("roc_auc no teste, 20 sementes de divisao, mesma tabela e mesmo modelo")
plt.ylabel("roc_auc")
plt.show()


Mesma tabela, mesmo modelo, mesma proporção de treino e teste: só a semente muda, e a nota vai de 0,9883
a 0,9996, uma amplitude de 0,0113. Mais de um centésimo, e isso é maior do que a diferença entre muitos
modelos que as pessoas comparam em artigo. Reportar a nota de uma semente, sem dizer qual, é reportar um
ponto sorteado.

## Shufflar sem estratificar, e sem shufflar

Dois erros que mudam tudo, e o alvo acima mostra o caminho. O primeiro: **não embaralhar** com o alvo
ordenado. O segundo: embaralhar sem **estratificar**, o que em dado desbalanceado deixa a proporção das
classes variar de fold para fold.


In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
positivos = np.flatnonzero(y == 1)
negativos = np.flatnonzero(y == 0)
desbalanceada = np.concatenate([negativos, positivos[: max(1, int(0.10 * len(positivos)))]])
X_des, y_des = X.iloc[desbalanceada].reset_index(drop=True), y.iloc[desbalanceada].reset_index(drop=True)
print(f"tabela desbalanceada: {len(X_des)} linhas, {y_des.mean():.1%} de classe positiva")

protocolos = {
    "KFold sem embaralhar": KFold(10),
    "KFold embaralhado": KFold(10, shuffle=True, random_state=RANDOM_STATE),
    "StratifiedKFold": StratifiedKFold(10, shuffle=True, random_state=RANDOM_STATE),
}
print()
notas_desbalanceada = {}
for nome, protocolo in protocolos.items():
    taxas = [y_des.to_numpy()[teste].mean() for _, teste in protocolo.split(X_des, y_des)]
    vazios = sum(taxa == 0 for taxa in taxas)
    notas = cross_val_score(modelo, X_des, y_des, cv=protocolo, scoring="roc_auc")
    notas_desbalanceada[nome] = notas
    validos = int(np.sum(~np.isnan(notas)))
    print(f"{nome:22s} positivos por fold: {min(taxas):5.1%} a {max(taxas):5.1%} "
          f"| folds sem nenhum positivo: {vazios} | folds com metrica valida: {validos}/10 "
          f"| media dos validos: {np.nanmean(notas):.4f}")
    print(f"{'':22s} taxas por fold: {[round(float(t), 3) for t in taxas]}")


O primeiro protocolo quebra na frente: sem embaralhar, com o alvo ordenado, **8 dos 10 folds ficaram sem
nenhum positivo** e só 1 produziu métrica. A média dos folds válidos deu 1,0000, um número perfeito vindo
de um protocolo destruído, e é aí que mora o perigo: quem calcula `nanmean` e não olha quantos folds
sobreviveram publica 1,0000 sem perceber que a medição não existiu.

Embaralhando sem estratificar, os folds ficam entre 4% e 28% de positivos, sete vezes de variação, e a
nota fica em 0,9874 contra 0,9948 do estratificado. Nesta tabela a diferença é pequena, e ela cresce
conforme a classe rara fica mais rara: com 1% de positivos, algum fold pode acabar sem nenhum.

As regras: **embaralhar é obrigatório quando a tabela veio ordenada**, **estratificar é o padrão em
classificação**, e **olhar quantos folds sobreviveram faz parte de ler a métrica**.

## Quando as linhas não são independentes

Aqui está o erro mais caro dos cinco, porque ele não gera aviso nenhum. Se uma mesma pessoa (paciente,
cliente, sensor) aparece em várias linhas, e as colunas dela se repetem entre as linhas, então embaralhar
coloca linhas da mesma pessoa no treino e no teste. O modelo não aprende o problema: ele reconhece a
pessoa.

A tabela abaixo tem 120 pessoas, 5 linhas cada, duas colunas que são cadastro da pessoa, repetidas em
todas as linhas dela, e um rótulo **sorteado por pessoa, sem relação alguma com essas colunas**. Como o
rótulo não depende de nada, a nota honesta é 0,5.


In [ ]:
n_pessoas, linhas_por_pessoa = 120, 5
pessoa = np.repeat(np.arange(n_pessoas), linhas_por_pessoa)
n = len(pessoa)

# duas colunas que são cadastro da pessoa, repetidas em todas as linhas dela
X_grupo = pd.DataFrame({
    "plano_do_cliente": rng.normal(0, 1.0, n_pessoas)[pessoa],
    "canal_de_aquisicao": rng.normal(0, 1.0, n_pessoas)[pessoa],
    "acessos_no_mes": rng.poisson(5, n),
})
rotulo_da_pessoa = rng.integers(0, 2, n_pessoas)   # sorteado por pessoa, sem relação com as colunas
y_grupo = pd.Series(rotulo_da_pessoa[pessoa], name="alvo")

print(f"tabela: {n} linhas de {n_pessoas} pessoas, alvo {y_grupo.mean():.1%}")
print("o rotulo e sorteado por pessoa, sem relacao com nenhuma coluna: a nota honesta e 0,5")
print()
notas_grupo = {}
linhas_grupo = []
for nome_modelo, classificador in {
    "regressao logistica": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "kNN com 1 vizinho": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=1)),
}.items():
    for nome_protocolo, protocolo in {
        "KFold 5 (ignora a pessoa)": KFold(5, shuffle=True, random_state=RANDOM_STATE),
        "GroupKFold 5 (pessoa inteira)": GroupKFold(5),
    }.items():
        if isinstance(protocolo, GroupKFold):
            notas = cross_val_score(classificador, X_grupo, y_grupo, cv=protocolo, groups=pessoa,
                                    scoring="roc_auc")
        else:
            notas = cross_val_score(classificador, X_grupo, y_grupo, cv=protocolo, scoring="roc_auc")
        notas_grupo[(nome_modelo, nome_protocolo)] = notas
        linhas_grupo.append({"modelo": nome_modelo, "protocolo": nome_protocolo,
                             "roc_auc": round(notas.mean(), 4), "desvio": round(notas.std(), 4)})

display(pd.DataFrame(linhas_grupo).set_index(["modelo", "protocolo"]))
print("a nota honesta é 0,5: o rótulo não depende de nada que esteja na tabela")


O kNN de um vizinho confirma o estrago: 0,7141 embaralhando contra 0,5140 agrupando. Vinte pontos de
AUC que não existem, num dado onde não há sinal nenhum.

E tem um detalhe na tabela acima que vale mais que o número: a **regressão logística não achou o
vazamento** (0,5048 contra 0,4678, os dois perto de 0,5, que é o correto). Um modelo linear não consegue
decorar 120 rótulos arbitrários; o de vizinho mais próximo consegue. O vazamento por grupo depende do
protocolo **e** do modelo, e é por isso que ele passa despercebido: o protocolo está errado, e o modelo
que você escolheu por acaso não tem como mostrar.

A pergunta que evita esse erro é sempre a mesma: **esta linha é independente das outras?** Se um cliente,
um paciente ou um dia de coleta responde mais de uma linha, o `groups=` entra na validação.

## Quando existe ordem no tempo

O terceiro erro que não gera aviso: embaralhar série temporal. Numa série, as linhas vizinhas são
parecidas, então embaralhar coloca no treino justamente os vizinhos que respondem pelo teste. O modelo
interpola, e a nota fica alta por um motivo que não existe no dia da previsão.


In [ ]:
n_tempo = 300
tempo = np.arange(n_tempo)
serie = 20 + 0.08 * tempo + 3 * np.sin(tempo / 12) + rng.normal(0, 0.6, n_tempo)

X_tempo = pd.DataFrame({
    "tempo": tempo,
    "mes_do_ano": tempo % 12,
    "valor_anterior": np.concatenate([[serie[0]], serie[:-1]]),
})
y_tempo = pd.Series(serie, name="valor")

regressao = make_pipeline(StandardScaler(), LinearRegression())

notas_kfold = cross_val_score(regressao, X_tempo, y_tempo, cv=KFold(5, shuffle=True,
                                                                   random_state=RANDOM_STATE), scoring="r2")
notas_temporal = cross_val_score(regressao, X_tempo, y_tempo, cv=TimeSeriesSplit(5), scoring="r2")
print(f"KFold embaralhado: r2 medio {notas_kfold.mean():.4f} (desvio {notas_kfold.std():.4f})")
print(f"TimeSeriesSplit:   r2 medio {notas_temporal.mean():.4f} (desvio {notas_temporal.std():.4f})")
print(f"notas do TimeSeriesSplit, fold a fold: {[round(n, 3) for n in notas_temporal]}")


O `KFold` embaralhado dá 0,9861 de R², e é a nota falsa: com `valor_anterior` na tabela, cada linha de
teste tem os vizinhos no treino, e a série vira interpolação. O `TimeSeriesSplit` treina sempre no
passado e testa no futuro, que é o que vai acontecer de verdade, e a nota cai para 0,7304, com um fold em
0,515. A distância entre as duas é o tamanho da mentira.

Regra de série temporal: **o teste é o futuro, e nenhuma linha futura entra no treino**. A validação passa
a ser por origem deslizante, que é o backtesting do notebook de série temporal.

## Mais de uma métrica, e o custo de medir

`cross_validate` devolve várias métricas e o tempo de cada ajuste, o que evita rodar a validação cinco
vezes para ver cinco números.


In [ ]:
t0 = time.perf_counter()
resultados = cross_validate(modelo, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                            scoring=["roc_auc", "accuracy", "f1"], return_train_score=True)
tempo = time.perf_counter() - t0

resumo_metricas = pd.DataFrame({
    "metrica": ["roc_auc", "accuracy", "f1"],
    "no teste": [resultados["test_roc_auc"].mean(), resultados["test_accuracy"].mean(),
                 resultados["test_f1"].mean()],
    "desvio": [resultados["test_roc_auc"].std(), resultados["test_accuracy"].std(),
               resultados["test_f1"].std()],
    "no treino": [resultados["train_roc_auc"].mean(), resultados["train_accuracy"].mean(),
                  resultados["train_f1"].mean()],
}).set_index("metrica").round(4)
display(resumo_metricas)
print(f"custo: {tempo:.1f}s para 5 folds com 3 metricas | tempo medio de ajuste: "
      f"{resultados['fit_time'].mean():.3f}s")
print("a coluna 'no treino' existe para mostrar a distancia entre treino e teste, que é o overfitting")


## O número que escolhe e o número que reporta

O último erro é o mais sutil e o mais comum em trabalho publicado: escolher o hiperparâmetro olhando a
validação cruzada e depois reportar a mesma validação cruzada. A escolha já viu todos os folds, então o
número está contaminado pelo processo de escolha.

A validação cruzada aninhada separa as coisas: um laço externo mede, e um laço interno escolhe dentro de
cada fold externo.


In [ ]:
bruta = make_classification(n_samples=350, n_features=40, n_informative=3, n_redundant=0,
                            class_sep=0.6, flip_y=0.25, random_state=RANDOM_STATE)
X_ruido = pd.DataFrame(bruta[0])
y_ruido = pd.Series(bruta[1], name="alvo")
X_treino_n, X_intocado, y_treino_n, y_intocado = train_test_split(
    X_ruido, y_ruido, test_size=0.3, stratify=y_ruido, random_state=RANDOM_STATE)
print(f"tabela com ruído: {len(X_ruido)} linhas, {X_ruido.shape[1]} colunas, "
      f"{y_ruido.mean():.1%} de classe positiva")
print("tabela ruim de propósito: 3 colunas informativas em 40, e 25% dos rótulos trocados")
print()

para_buscar = make_pipeline(StandardScaler(), SelectKBest(f_classif), LogisticRegression(max_iter=5000))
grade = {"selectkbest__k": [2, 5, 10, 20, 40],
         "logisticregression__C": [0.01, 0.1, 1, 10, 100]}
busca = GridSearchCV(para_buscar, grade, cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                     scoring="roc_auc")
busca.fit(X_treino_n, y_treino_n)
nota_do_melhor = busca.best_score_                      # escolha e medicao na mesma particao
aninhado = cross_val_score(busca, X_treino_n, y_treino_n,
                           cv=StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE),
                           scoring="roc_auc")
nota_intocada = roc_auc_score(y_intocado, busca.predict_proba(X_intocado)[:, 1])

print(f"configuracoes testadas: {len(busca.cv_results_['mean_test_score'])} "
      f"| melhor: k={busca.best_params_['selectkbest__k']}, C={busca.best_params_['logisticregression__C']}")
print(f"nota do GridSearchCV (escolhe e mede no mesmo lugar): {nota_do_melhor:.4f}")
print(f"validacao aninhada (escolhe dentro de cada fold):     {aninhado.mean():.4f} "
      f"(desvio {aninhado.std():.4f})")
print(f"tabela intocada, olhada uma vez no fim:               {nota_intocada:.4f}")
print(f"distancia entre o numero do GridSearchCV e a tabela intocada: "
      f"{nota_do_melhor - nota_intocada:.4f}")


Três números, na mesma tabela ruim de propósito, 3 colunas informativas em 40 e 25% dos rótulos trocados,
com 25 configurações testadas:

- o **GridSearchCV** escolhe e mede nas mesmas partições: 0,7550, otimista por construção;
- a **validação aninhada**, em que a escolha acontece dentro de cada fold: 0,7359, a estimativa honesta;
- a **tabela intocada**, olhada uma vez no fim: 0,8280, que engana para o outro lado, porque 105 linhas
  produzem uma medição ruidosa.

A diferença de 0,019 entre o primeiro e o segundo é da mesma ordem das diferenças entre modelos que se
veem por aí, e é dinheiro que não existe. A lição do terceiro número é complementar: uma tabela intocada
pequena também não é árbitro. O que sustenta uma afirmação é o protocolo, não um número solto.

## O resumo dos protocolos, nos mesmos dados

Juntando tudo numa tabela, com o que cada protocolo mede de verdade.


In [ ]:
linhas_resumo = [
    {"onde": "cancer de mama, 20 sementes", "protocolo": "uma divisao so (media das 20)",
     "metrica": "roc_auc", "valor": notas_de_uma_divisao.mean(), "desvio": notas_de_uma_divisao.std()},
    {"onde": "desbalanceada", "protocolo": "KFold 10 sem embaralhar",
     "metrica": "roc_auc", "valor": np.nanmean(notas_desbalanceada["KFold sem embaralhar"]),
     "desvio": np.nanstd(notas_desbalanceada["KFold sem embaralhar"])},
    {"onde": "desbalanceada", "protocolo": "StratifiedKFold 10",
     "metrica": "roc_auc", "valor": np.nanmean(notas_desbalanceada["StratifiedKFold"]),
     "desvio": np.nanstd(notas_desbalanceada["StratifiedKFold"])},
    {"onde": "120 pessoas, 5 linhas cada, sem sinal", "protocolo": "kNN 1 vizinho, KFold 5",
     "metrica": "roc_auc",
     "valor": notas_grupo[("kNN com 1 vizinho", "KFold 5 (ignora a pessoa)")].mean(),
     "desvio": notas_grupo[("kNN com 1 vizinho", "KFold 5 (ignora a pessoa)")].std()},
    {"onde": "120 pessoas, 5 linhas cada, sem sinal", "protocolo": "kNN 1 vizinho, GroupKFold 5",
     "metrica": "roc_auc",
     "valor": notas_grupo[("kNN com 1 vizinho", "GroupKFold 5 (pessoa inteira)")].mean(),
     "desvio": notas_grupo[("kNN com 1 vizinho", "GroupKFold 5 (pessoa inteira)")].std()},
    {"onde": "serie temporal", "protocolo": "KFold 5 embaralhado (errado)",
     "metrica": "r2", "valor": notas_kfold.mean(), "desvio": notas_kfold.std()},
    {"onde": "serie temporal", "protocolo": "TimeSeriesSplit 5 (certo)",
     "metrica": "r2", "valor": notas_temporal.mean(), "desvio": notas_temporal.std()},
    {"onde": "tabela com ruido", "protocolo": "GridSearchCV (escolhe e mede no mesmo lugar)",
     "metrica": "roc_auc", "valor": nota_do_melhor, "desvio": float("nan")},
    {"onde": "tabela com ruido", "protocolo": "validacao aninhada",
     "metrica": "roc_auc", "valor": aninhado.mean(), "desvio": aninhado.std()},
    {"onde": "tabela com ruido", "protocolo": "tabela intocada (olhada uma vez)",
     "metrica": "roc_auc", "valor": nota_intocada, "desvio": float("nan")},
]
display(pd.DataFrame(linhas_resumo).set_index(["onde", "protocolo"]).round(4))


## Como ler isso

1. **uma divisão só não é medição**: rode várias sementes e reporte média e desvio;
2. **embaralhe** se a tabela veio em ordem, e **estratifique** em classificação;
3. pergunte se a linha é independente: se existe grupo, use `groups=`;
4. se existe tempo, o teste é o futuro: `TimeSeriesSplit`, nunca embaralhar;
5. se o número serviu para escolher, use validação aninhada para o número que você reporta;
6. sempre diga o protocolo exato: semente, número de folds, o que foi estratificado e o que foi agrupado.

## Resumo

A validação cruzada não é uma formalidade no fim do notebook: é a decisão que define se o número tem
sentido. Neste notebook: uma divisão só varia alguns centésimos entre sementes; sem embaralhar, folds
ficam sem a classe positiva e a métrica não existe; sem estratificar, a proporção de positivos oscila
entre folds; ignorando o grupo, a nota sobe de 0,5 para quase 1,0 em dado onde não há sinal nenhum; e
embaralhando série temporal, a nota também sobe por um motivo que não existirá na previsão.

**Próximo passo:** o notebook de seleção de features desta pasta, que usa esta régua para comparar
métodos de escolha de coluna.

## Referências

```bibtex
@inproceedings{kohavi1995study,
  title     = {A Study of Cross-Validation and Bootstrap for Accuracy Estimation and Model Selection},
  author    = {Kohavi, Ron},
  booktitle = {Proceedings of the 14th International Joint Conference on Artificial Intelligence},
  volume    = {2},
  pages     = {1137--1143},
  year      = {1995}
}

@article{varma2006bias,
  title   = {Bias in Error Estimation When Using Cross-Validation for Model Selection},
  author  = {Varma, Sudhir and Simon, Richard},
  journal = {BMC Bioinformatics},
  volume  = {7},
  number  = {1},
  pages   = {91},
  year    = {2006},
  doi     = {10.1186/1471-2105-7-91}
}

@article{roberts2017cross,
  title   = {Cross-Validation Strategies for Data with Temporal, Spatial, Hierarchical, or
             Phylogenetic Structure},
  author  = {Roberts, David R. and Bahn, Volker and Ciuti, Simone and Boyce, Mark S. and
             Elith, Jane and Guillera-Arroita, Gurutzeta and Hauenstein, Severin and
             Lahoz-Monfort, Jos{'e} J. and Schr{"o}der, Boris and Thuiller, Wilfried and
             Warton, David I. and Wintle, Brendan A. and Hartig, Florian and Dormann, Carsten F.},
  journal = {Ecography},
  volume  = {40},
  number  = {8},
  pages   = {913--929},
  year    = {2017},
  doi     = {10.1111/ecog.02881}
}
```
